# Make background SNVs
Background SNVs are natural SNVs added to variant-effect training (`--mode variant_effect` / `variant_effect_twinnet`) alongside the synthetic variants. Their effect sizes come from GTEx v8 eQTL summary statistics.

Download the GTEx v8 all-associations file for your tissue (e.g. `Whole_Blood.v8.allpairs.parquet`) from the GTEx portal (https://gtexportal.org/home/downloads/adult-gtex/qtl).

Each background SNV gets `background_delta = 2 * slope`. GTEx slopes are per copy of the alt allele. In out our training setup, each variant is placed on both maternal and paternal haplotypes, so we model its effect as twice the per-allele slope.

In [ ]:
TISSUE = 'Whole_Blood'
WINDOW_SIZE = 49152  # bp, TSS-centered. Must match seq_len in the training config
ALLPAIRS = f'{TISSUE}.v8.allpairs.parquet'
MOTIF_POSITIONS_CSV = '../../data/motif_positions/STAT1_uniform_motif_positions_49kb.csv'
OUT_CSV = f'../../data/background_snvs/background_snvs_{TISSUE}.csv.gz'

In [ ]:
import os
import pandas as pd

DATA_DIR = '../../data'

## Train genes and their windows
Background SNVs are only used for training, so only train genes are kept.

In [ ]:
positions = pd.read_csv(MOTIF_POSITIONS_CSV)
positions = positions[positions['set'] == 'train']
assert all(positions['short_end0'] - positions['short_start0'] == WINDOW_SIZE)
gene_map = pd.read_csv(os.path.join(DATA_DIR, 'gencode_v26_gene_id_to_name.tsv'), sep='\t')

## SNVs inside each window
SNVs that overlap the gene's synthetic-variant motif are dropped.

In [ ]:
eqtl = pd.read_parquet(ALLPAIRS, columns=['gene_id', 'variant_id', 'slope'])
eqtl = eqtl.merge(gene_map, on='gene_id')
eqtl = eqtl[eqtl['gene_name'].isin(positions['gene_name'])]

variant = eqtl['variant_id'].str.split('_', expand=True)  # chr_pos_ref_alt_b38, pos is 1-based
eqtl['chrom'] = variant[0]
eqtl['pos0'] = variant[1].astype(int) - 1
eqtl['other_allele'] = variant[2]
eqtl['effect_allele'] = variant[3]
eqtl = eqtl[(eqtl['other_allele'].str.len() == 1) & (eqtl['effect_allele'].str.len() == 1)]  # SNVs only

eqtl = eqtl.merge(positions[['gene_name', 'chrom', 'short_start0', 'short_end0', 'chromStart', 'chromEnd', 'set']], on=['gene_name', 'chrom'])
eqtl = eqtl[(eqtl['pos0'] >= eqtl['short_start0']) & (eqtl['pos0'] < eqtl['short_end0'])]
eqtl = eqtl[(eqtl['pos0'] < eqtl['chromStart']) | (eqtl['pos0'] >= eqtl['chromEnd'])]
eqtl['synthetic_snp_position'] = eqtl['pos0'] - eqtl['short_start0']  # position within the input sequence
eqtl['background_delta'] = 2 * eqtl['slope']
eqtl['snp_type'] = 'background'
eqtl = eqtl.rename(columns={'variant_id': 'var_id'})
print(eqtl.groupby('gene_name').size().describe())

In [ ]:
os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)
eqtl[['gene_name', 'var_id', 'chrom', 'pos0', 'synthetic_snp_position', 'other_allele', 'effect_allele', 'background_delta', 'set', 'snp_type']].to_csv(OUT_CSV, index=False)